# Streaming Agent Responses

Waiting for an agent to finish before showing any output makes it feel slow and opaque. Streaming step events lets users see the agent's reasoning in real time — which tool it chose, what it observed, and how its thinking evolved toward a final answer.

**What you'll build:** a streaming agent display that shows thoughts, tool calls, observations, and tokens as they arrive — suitable for both terminal output and real-time web UIs.

**Time:** ~15 min | **Difficulty:** Beginner

**What you'll learn:**
- The difference between `agent.run()` (blocking) and `agent.stream_steps()` (streaming)
- All six `StepEvent` types: `ThoughtEvent`, `ActionEvent`, `ObservationEvent`, `TokenEvent`, `FinalAnswerEvent`, `ErrorEvent`
- How to build a formatted terminal display for step events
- How `ReActAgent` and `FunctionCallingAgent` differ in what events they emit
- How to pipe events to a WebSocket or SSE endpoint

## Prerequisites

You need:
- An **OpenAI API key** (`OPENAI_API_KEY`)
- The `synapsekit` package (installed below)

In [ ]:
!pip install synapsekit -q

In [ ]:
import os

# Set your API key here
# os.environ['OPENAI_API_KEY'] = 'sk-...'

## Step 1: Import event types

Import both agent types and all six event types.

In [ ]:
import asyncio
from synapsekit.agents import (
    ReActAgent,
    FunctionCallingAgent,
    DuckDuckGoSearchTool,
    CalculatorTool,
    WikipediaTool,
    # Step event types
    ActionEvent,
    ErrorEvent,
    FinalAnswerEvent,
    ObservationEvent,
    ThoughtEvent,
    TokenEvent,
)
from synapsekit.llms.openai import OpenAILLM

## Step 2: Understand the event types

`stream_steps()` yields a union type `StepEvent`. Each concrete type carries different fields:

| Event | Fields | When emitted |
|---|---|---|
| `ThoughtEvent` | `thought: str` | LLM produces a Thought line (ReActAgent only) |
| `ActionEvent` | `tool: str`, `tool_input` | LLM decides to call a tool |
| `ObservationEvent` | `observation: str`, `tool: str` | Tool execution completes |
| `TokenEvent` | `token: str` | A single text token from the LLM |
| `FinalAnswerEvent` | `answer: str` | Agent has a complete final answer |
| `ErrorEvent` | `error: str` | A tool raised an exception |

**Note:** `ReActAgent` emits `ThoughtEvent` + `TokenEvent`; `FunctionCallingAgent` emits `ActionEvent` + `ObservationEvent` + `TokenEvent` (final answer only) + `FinalAnswerEvent`.

In [ ]:
# Verify event types are importable and distinct
event_types = [ThoughtEvent, ActionEvent, ObservationEvent, TokenEvent, FinalAnswerEvent, ErrorEvent]
print("Event types available:")
for et in event_types:
    print(f"  {et.__name__}")

## Step 3: Build a terminal renderer

A simple renderer maps each event type to a distinct visual prefix, making the agent's reasoning easy to follow at a glance.

In [ ]:
COLORS = {
    "thought":      "\033[36m",  # cyan
    "action":       "\033[33m",  # yellow
    "observation":  "\033[32m",  # green
    "token":        "\033[0m",   # default
    "final_answer": "\033[1;32m",# bold green
    "error":        "\033[31m",  # red
}
RESET = "\033[0m"


def render_event(event) -> None:
    if isinstance(event, ThoughtEvent):
        print(f"{COLORS['thought']}Thought:{RESET} {event.thought}")
    elif isinstance(event, ActionEvent):
        input_preview = str(event.tool_input)[:80]
        print(f"{COLORS['action']}Action:{RESET}  {event.tool}({input_preview})")
    elif isinstance(event, ObservationEvent):
        preview = event.observation[:150]
        print(f"{COLORS['observation']}Result:{RESET}  {preview}{'...' if len(event.observation) > 150 else ''}")
    elif isinstance(event, TokenEvent):
        # Print tokens inline without newline — forms a continuous stream
        print(event.token, end="", flush=True)
    elif isinstance(event, FinalAnswerEvent):
        print(f"\n{COLORS['final_answer']}Answer:{RESET}\n{event.answer}")
    elif isinstance(event, ErrorEvent):
        print(f"{COLORS['error']}Error:{RESET}   {event.error}")

print("Terminal renderer defined.")

## Step 4: Stream a ReActAgent

`ReActAgent` streams LLM tokens live as they are generated. The full response is then parsed for Thought/Action/Final Answer structure. This means you see the LLM "typing" in real time.

In [ ]:
react_agent = ReActAgent(
    llm=OpenAILLM(model="gpt-4o-mini"),
    tools=[DuckDuckGoSearchTool(), WikipediaTool(), CalculatorTool()],
    max_iterations=6,
)

async def stream_react(question: str) -> None:
    print(f"Q: {question}\n")
    async for event in react_agent.stream_steps(question):
        render_event(event)

asyncio.run(stream_react("What is the population of Tokyo according to Wikipedia?"))

## Step 5: Stream a FunctionCallingAgent

`FunctionCallingAgent` does not stream raw tokens during tool-selection turns — the LLM response is processed as a complete JSON object. It emits `TokenEvent` for the final answer only. The trade-off is more reliable tool parsing.

In [ ]:
fc_agent = FunctionCallingAgent(
    llm=OpenAILLM(model="gpt-4o-mini"),
    tools=[DuckDuckGoSearchTool(), CalculatorTool()],
    system_prompt="You are a helpful research assistant.",
    max_iterations=6,
)

async def stream_fc(question: str) -> None:
    print(f"Q: {question}\n")
    async for event in fc_agent.stream_steps(question):
        render_event(event)

asyncio.run(stream_fc("What is 15% of 2450?"))

## Step 6: Collect the full answer from the stream

When you need both the streaming display and the final answer string, accumulate `FinalAnswerEvent`.

In [ ]:
async def stream_and_collect(agent, question: str) -> str:
    final_answer = ""
    async for event in agent.stream_steps(question):
        render_event(event)
        if isinstance(event, FinalAnswerEvent):
            final_answer = event.answer
    return final_answer

answer = asyncio.run(stream_and_collect(fc_agent, "How many days are in a leap year?"))
print(f"\nCollected answer: {answer[:100]}")

## Complete Working Example

A self-contained script demonstrating both `ReActAgent` and `FunctionCallingAgent` streaming with the same renderer function.

In [ ]:
import asyncio
from synapsekit.agents import (
    ActionEvent,
    CalculatorTool,
    DuckDuckGoSearchTool,
    ErrorEvent,
    FinalAnswerEvent,
    FunctionCallingAgent,
    ObservationEvent,
    ReActAgent,
    ThoughtEvent,
    TokenEvent,
    WikipediaTool,
)
from synapsekit.llms.openai import OpenAILLM

RESET = "\033[0m"


def render_event(event) -> None:
    if isinstance(event, ThoughtEvent):
        print(f"\033[36m[thought]\033[0m {event.thought}")
    elif isinstance(event, ActionEvent):
        print(f"\033[33m[tool]\033[0m    {event.tool}({str(event.tool_input)[:70]})")
    elif isinstance(event, ObservationEvent):
        print(f"\033[32m[result]\033[0m  {event.observation[:120]}...")
    elif isinstance(event, TokenEvent):
        print(event.token, end="", flush=True)
    elif isinstance(event, FinalAnswerEvent):
        print(f"\n\033[1;32m[answer]\033[0m\n{event.answer}")
    elif isinstance(event, ErrorEvent):
        print(f"\033[31m[error]\033[0m   {event.error}")


async def demo_react(question: str) -> None:
    agent = ReActAgent(
        llm=OpenAILLM(model="gpt-4o-mini"),
        tools=[DuckDuckGoSearchTool(), WikipediaTool(), CalculatorTool()],
        max_iterations=6,
    )
    print(f"\n--- ReActAgent ---\nQ: {question}\n")
    async for event in agent.stream_steps(question):
        render_event(event)


async def demo_function_calling(question: str) -> None:
    agent = FunctionCallingAgent(
        llm=OpenAILLM(model="gpt-4o-mini"),
        tools=[DuckDuckGoSearchTool(), CalculatorTool()],
        system_prompt="You are a concise assistant.",
        max_iterations=6,
    )
    print(f"\n--- FunctionCallingAgent ---\nQ: {question}\n")
    async for event in agent.stream_steps(question):
        render_event(event)


async def main() -> None:
    await demo_react(
        "What is the population of Tokyo according to Wikipedia, and what is 10% of that number?"
    )
    await demo_function_calling(
        "Search for the latest Python release version and calculate how many months ago it was released from today (April 2026)."
    )


asyncio.run(main())

## Next Steps

- [ReAct Research Assistant](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/react-research-assistant) — build a full research agent with streaming step display
- [Multi-Tool Orchestration](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/multi-tool-orchestration) — stream a complex multi-tool pipeline
- [Structured Output with Function Calling](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/structured-output-function-calling) — enforce typed output schemas while still streaming intermediate steps